# Importing the necessary packages/modules

In [25]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException


In [ ]:
import pandas as pd
import time
import os
import random

# Getting familiar with selenium and navigating the new website UI

In [ ]:

driver = webdriver.Chrome()

driver.get("https://cwlagos.com/property")
# this gives me the details of the first property on the page, but I want to get all the properties
elems = driver.find_elements(By.CLASS_NAME, "framer-12de3j-container")
listings = []
for i in range(len(elems)):
    listings.append({
        "No": i,
        "Details": elems[i].text
    })
print(listings)
driver.quit()

In [4]:
print(listings[0]['Details'].split('\n'))

['For Rent', 'Commercial', '$600', 'GRADE A COMMERCIAL BUILDING– IKOYI', 'Ikoyi', 'Nadi', '+2349062511345']


In [ ]:
# now i need to work on clicking the "load more" button to get more listings
driver = webdriver.Chrome()
driver.get("https://cwlagos.com/property")

# a 'wait' object so I don't need to use time.sleep() everywhere
wait = WebDriverWait(driver, 10)

while True:
    try:
        # Try to find and click the "Load More" button
        load_more = wait.until(EC.element_to_be_clickable((By.XPATH, "//*[contains(text(), 'Load')]")))
        load_more.click()
        time.sleep(2)  # Wait for new content to load
        driver.execute_script("window.scrollBy(0, 600);")
        
    except Exception as e:
        # If the button doesn't exist or can't be clicked, we're done
        print(f"No more 'Load More' button found: {e}")
        break

# Now that all content is loaded, scrape everything
print("All listings loaded. Scraping...")
elems = driver.find_elements(By.CLASS_NAME, "framer-12de3j-container")

listings = []
for i, elem in enumerate(elems):
    listings.append({
        "No": i+1,
        "Details": elem.text
    })
driver.quit()
print(f"Found {len(listings)} listings total.")

In [ ]:
listings[0]['Details'].split('\n')

In [ ]:
listings = []
for i, elem in enumerate(elems):
    inlist = elem.text.split('\n')
    row ={}
    for i in range(len(inlist)):
        row[i+1] = inlist[i]
    listings.append(row)
    
# vague logic and is very likely to make the data unintelliiblle. sigh!


# Creating a python script that navigates the site, finds more content and stores them for parsing

In [ ]:
def parse_listing(elem):
    """Helper function for parsing a single listing card by finding specific elements within it..."""
    listing = {
        'type': None,
        'kind': None,
        'price': None,
        'title': None,
        'location': None,
        'beds': None,
        'baths': None,
        'agent': None,
        'contact': None
    }
    
    try:
        listing['type'] = elem.find_element(By.CLASS_NAME, "framer-cecu2t-container").text.strip()
    except:
        pass
    
    try:
        listing['kind'] = elem.find_element(By.CLASS_NAME, "framer-qv6gk7-container").text.strip()
    except:
        pass
    
    try:
        listing['price'] = elem.find_element(By.CLASS_NAME, "framer-maw6ss").text.strip()
    except:
        pass
    
    try:
        listing['title'] = elem.find_element(By.CLASS_NAME, "framer-1aak7at").text.strip()
    except:
        pass
    
    try:
        listing['location'] = elem.find_element(By.CLASS_NAME, "framer-gwir1w").text.strip()
    except:
        pass
    
    try:
        listing['beds'] = elem.find_element(By.CLASS_NAME, "framer-ix3jh1").text.strip()
    except:
        pass
    
    try:
        listing['baths'] = elem.find_element(By.CLASS_NAME, "framer-16vvyc2").text.strip()
    except:
        pass
    
    try:
        listing['agent'] = elem.find_element(By.CLASS_NAME, "framer-1kvgq0a").text.strip()
    except:
        pass
    
    try:
        listing['contact'] = elem.find_element(By.CLASS_NAME, "framer-9ce212").text.strip()
    except:
        pass
    
    return listing


In [18]:

driver = webdriver.Chrome()
driver.get("https://cwlagos.com/property")

# a 'wait' object so I don't need to use time.sleep() everywhere
wait = WebDriverWait(driver, 10)

while True:
    try:
        # Try to find and click the "Load More" button
        load_more = wait.until(EC.element_to_be_clickable((By.XPATH, "//*[contains(text(), 'Load')]")))
        load_more.click()
        time.sleep(2)  # Wait for new content to load
        driver.execute_script("window.scrollBy(0, 600);")
        
    except Exception as e:
        # If the button doesn't exist or can't be clicked, we're done
        print(f"No more 'Load More' button found: {e}")
        break
# Now that all content is loaded, scrape everything
print("All listings loaded. Scraping...")


# main scraping loop:
elems = driver.find_elements(By.CLASS_NAME, "framer-12de3j-container")
listings = []

for elem in elems:
    parsed = parse_listing(elem)
    listings.append(parsed)

driver.quit()
print(f"Found {len(listings)} listings")
print(listings[0], listings[-1])

No more 'Load More' button found: Message: element click intercepted: Element <p class="framer-text framer-styles-preset-1yryb6d" data-styles-preset="B66GV9KSB" dir="auto" style="--framer-text-color:var(--extracted-r6o4lv, rgb(255, 255, 255))">...</p> is not clickable at point (405, 576). Other element would receive the click: <div class="framer-10qs1fj" data-framer-name="Main Properties" style="opacity: 1;">...</div>
  (Session info: chrome=147.0.7727.138); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementclickinterceptedexception
Stacktrace:
0   chromedriver                        0x00000001012cdebc cxxbridge1$str$ptr + 3213376
1   chromedriver                        0x00000001012c5e8c cxxbridge1$str$ptr + 3180560
2   chromedriver                        0x0000000100d8b8d4 _RNvCs10ygTOo3JCa_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 75096
3   chromedriver                        0x0000000100dd9570 _RNvC

In [19]:
print(len(listings))

486


In [ ]:
df = pd.DataFrame(listings)
output_path = '../data/raw/cwlagos_listings_0.csv'
df.to_csv(output_path, index=False)

## Attempt to get more listings from the site after making some tweaks to the scrape code

In [22]:

driver = webdriver.Chrome()
driver.get("https://cwlagos.com/property")

# a 'wait' object so I don't need to use time.sleep() everywhere
wait = WebDriverWait(driver, 10)

while True:
    try:
        # Try to find and click the "Load More" button
        load_more = wait.until(EC.element_to_be_clickable((By.XPATH, "//*[contains(text(), 'Load')]")))
        load_more.click()
        time.sleep(random.uniform(2, 4))  # Wait for new content to load
        driver.execute_script("window.scrollBy(0, 700);")
        time.sleep(1) 
        
    except Exception as e:
        # If the button doesn't exist or can't be clicked, we're done
        print(f"No more 'Load More' button found: {e}")
        break
# Now that all content is loaded, scrape everything
print("All listings loaded. Scraping...")


# main scraping loop:
elems = driver.find_elements(By.CLASS_NAME, "framer-12de3j-container")
listings = []

for elem in elems:
    parsed = parse_listing(elem)
    listings.append(parsed)

driver.quit()
print(f"Found {len(listings)} listings")
print(listings[0], listings[-1])

No more 'Load More' button found: Message: element click intercepted: Element <p class="framer-text framer-styles-preset-1yryb6d" data-styles-preset="B66GV9KSB" dir="auto" style="--framer-text-color:var(--extracted-r6o4lv, rgb(255, 255, 255))">...</p> is not clickable at point (405, 576). Other element would receive the click: <div class="framer-10qs1fj" data-framer-name="Main Properties" style="opacity: 1;">...</div>
  (Session info: chrome=147.0.7727.138); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementclickinterceptedexception
Stacktrace:
0   chromedriver                        0x0000000100a25ebc cxxbridge1$str$ptr + 3213376
1   chromedriver                        0x0000000100a1de8c cxxbridge1$str$ptr + 3180560
2   chromedriver                        0x00000001004e38d4 _RNvCs10ygTOo3JCa_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 75096
3   chromedriver                        0x0000000100531570 _RNvC

In [24]:
# I was able to get significantly more listings from the site this time
print(len(listings))

652


In [ ]:
df = pd.DataFrame(listings)
output_path = '../data/raw/cwlagos_listings_1.csv'
df.to_csv(output_path, index=False)

# Final attempt at scraping the listings, including the urls(unique identifier)

In [ ]:

def parse_listing_helper(elem):
    """Parse a single property listing card."""
    listing = {
        'type': None,
        'kind': None,
        'price': None,
        'title': None,
        'location': None,
        'beds': None,
        'baths': None,
        'agent': None,
        'contact': None,
        'listing_url': None
    }
    
    try:
        listing['type']     = elem.find_element(By.CLASS_NAME, "framer-cecu2t-container").text.strip()
    except NoSuchElementException:
        pass
    try:
        listing['kind']     = elem.find_element(By.CLASS_NAME, "framer-qv6gk7-container").text.strip()
    except NoSuchElementException:
        pass
    try:
        listing['price']    = elem.find_element(By.CLASS_NAME, "framer-maw6ss").text.strip()
    except NoSuchElementException:
        pass
    try:
        listing['title']    = elem.find_element(By.CLASS_NAME, "framer-1aak7at").text.strip()
    except NoSuchElementException:
        pass
    try:
        listing['location'] = elem.find_element(By.CLASS_NAME, "framer-gwir1w").text.strip()
    except NoSuchElementException:
        pass
    try:
        listing['beds']     = elem.find_element(By.CLASS_NAME, "framer-ix3jh1").text.strip()
    except NoSuchElementException:
        pass
    try:
        listing['baths']    = elem.find_element(By.CLASS_NAME, "framer-16vvyc2").text.strip()
    except NoSuchElementException:
        pass
    try:
        listing['agent']    = elem.find_element(By.CLASS_NAME, "framer-1kvgq0a").text.strip()
    except NoSuchElementException:
        pass
    try:
        listing['contact']  = elem.find_element(By.CLASS_NAME, "framer-9ce212").text.strip()
    except NoSuchElementException:
        pass
    try:
        link_elem = elem.find_element(By.TAG_NAME, "a")
        listing['listing_url'] = link_elem.get_attribute("href")
    except NoSuchElementException:
        pass
    return listing


def scrape_all_listings():
    options = webdriver.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    
    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 15)
    
    try:
        driver.get("https://cwlagos.com/property")
        time.sleep(3)  # Initial load
        
        print("Starting to click 'Load More' buttons...")
        
        while True:
            try:
                load_more = wait.until(
                    EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Load')] | //*[contains(text(), 'Load More')]"))
                )
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", load_more)
                time.sleep(random.uniform(1, 1.5))
                load_more.click()
                print("Clicked 'Load More'...")
                time.sleep(random.uniform(1, 2))
                
            except TimeoutException:
                print("No more 'Load More' button found. All listings loaded.")
                break
            except Exception as e:
                print(f"Error clicking load more: {e}")
                break
        
        print("Now extracting listings...")
        # Main container class you identified
        elems = driver.find_elements(By.CLASS_NAME, "framer-12de3j-container")
        listings = []
        
        for i, elem in enumerate(elems, 1):
            parsed = parse_listing_helper(elem)
            listings.append(parsed)
            if i % 50 == 0:
                print(f"Parsed {i}/{len(elems)} listings...")
        
        print(f"Successfully scraped {len(listings)} listings.")
        return listings
        
    finally:
        driver.quit()


def save_to_df(listings):
    df = pd.DataFrame(listings)
    df.to_csv("cwlagos_listings_raw.csv", index=False)
    print("Data saved to cwlagos_listings.csv")
    print(f"Total rows: {len(df)}")
    print(f"Columns: {list(df.columns)}")


    try:
        listings = scrape_all_listings()
        save_to_df(listings)
    except Exception as e:
        print(f"Script failed with error: {e}")
  